In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/colab_setup.py"

In [ ]:
import os
import sys
import glob
import json
import shutil
import re
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, Flatten, Concatenate, BatchNormalization,
    TimeDistributed, LSTM, Bidirectional
)
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import yaml
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from ml.notebooks.split_dataset import split_dataset

In [ ]:
# Install required packages if not already installed
try:
    import cv2
except ImportError:
    !pip install opencv-python-headless

try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm

# Process Data

In [ ]:
# Project paths setup
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to import from config
try:
    from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
except ModuleNotFoundError:
    print("Could not import config module. Adjusting path...")
    # Try with full path
    sys.path.insert(0, os.path.abspath(project_root))
    try:
        from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
    except ModuleNotFoundError:
        # Define default configuration if import fails
        print("Creating default configuration")

        def normalize_path(path):
            """Normalize a path to avoid duplicates caused by different representations"""
            if isinstance(path, str) and path.startswith('s3://'):
                return path  # Don't normalize S3 paths
            return os.path.normpath(path)

        # Default data directories
        base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
        output_dir = os.path.join(base_dir, "Data/processed")
        os.makedirs(output_dir, exist_ok=True)
        DATA_ROOT = normalize_path(os.path.join(base_dir, 'Data'))

        DATA_DIRS = {
            'raw': os.path.join(DATA_ROOT, 'Raw'),
            'stanford_original': os.path.join(DATA_ROOT, 'Raw', 'stanford_dog_pose'),
            'personal_dataset': os.path.join(DATA_ROOT, 'Raw', 'personal_dataset'),
            'matrix': os.path.join(DATA_ROOT, 'Matrix'),
            'interim': os.path.join(DATA_ROOT, 'interim'),
            'processed': os.path.join(DATA_ROOT, 'processed'),
            'models': os.path.join(base_dir, 'Models'),
        }

        # Default emotion mapping
        EMOTION_MAPPING = {
            "Happy or Playful": "Happy/Playful",
            "Relaxed": "Relaxed",
            "Submissive": "Submissive/Appeasement",
            "Excited": "Happy/Playful",
            "Drowsy or Bored": "Relaxed",
            "Curious or Confused": "Curiosity/Alertness",
            "Confident or Alert": "Curiosity/Alertness",
            "Jealous": "Stressed",
            "Stressed": "Stressed",
            "Frustrated": "Stressed",
            "Unsure or Uneasy": "Fearful/Anxious",
            "Possessive, Territorial, Dominant": "Aggressive/Threatening",
            "Fear or Aggression": "Fearful/Anxious",
            "Pain": "Stressed"
        }

        def ensure_directories():
            """Ensure all required directories exist"""
            for path in DATA_DIRS.values():
                if isinstance(path, str) and not path.startswith('s3://'):
                    Path(path).mkdir(parents=True, exist_ok=True)

            # Create subdirectories for processed data
            for split in ['train', 'validation', 'test']:
                split_path = os.path.join(DATA_DIRS['processed'], split)
                for subdir in ['images', 'annotations']:
                    path = os.path.join(split_path, subdir)
                    Path(path).mkdir(parents=True, exist_ok=True)

# Make sure directories exist
ensure_directories()

# Define configuration for the model (can be loaded from YAML if available)
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file or use defaults"""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    else:
        # Use default configuration
        print(f"Config file not found: {config_path}, using defaults")
        default_config = {
            "data": {
                "base_dir": project_root,
                "raw_data_dir": os.path.join(DATA_DIRS['raw']),
                "processed_data_dir": os.path.join(DATA_DIRS['processed']),
                "train_split": 0.7,
                "val_split": 0.15,
                "test_split": 0.15
            },
            "model": {
                "image_size": [224, 224, 3],
                "batch_size": 32,
                "learning_rate": 0.001,
                "dropout_rate": 0.5,
                "backbone": "mobilenetv2",
                "early_stopping_patience": 5
            },
            "training": {
                "checkpoint_dir": os.path.join(DATA_DIRS['models'], "checkpoints"),
                "logs_dir": os.path.join(DATA_DIRS['models'], "logs")
            },
            "inference": {
                "confidence_threshold": 0.6,
                "behavior_threshold": 0.5,
                "output_dir": os.path.join(DATA_DIRS['processed'], "predictions")
            }
        }

        # Save default config for future use
        os.makedirs(os.path.dirname(config_path), exist_ok=True)
        with open(config_path, 'w') as f:
            yaml.dump(default_config, f)

        return default_config

def load_config(config_path="/content/drive/MyDrive/Colab Notebooks/Pawnder/config.yaml"):
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        print(f"Successfully loaded config from {config_path}")
        return config
    except FileNotFoundError:
        print(f"Config file not found at: {config_path}")
        # Use the default config creation logic from the previous function
        return create_default_config(config_path)
    except PermissionError:
        print(f"Permission denied when trying to access: {config_path}")
        return create_default_config(config_path)
    except Exception as e:
        print(f"An error occurred when loading config: {str(e)}")
        return create_default_config(config_path)

# Define emotion classes from the EMOTION_MAPPING
CLASS_NAMES = list(set(EMOTION_MAPPING.values()))
# Safe class names for directories
SAFE_CLASS_NAMES = [emotion.replace("/", "_").replace("\\", "_") for emotion in CLASS_NAMES]
CLASS_MAP = dict(zip(CLASS_NAMES, SAFE_CLASS_NAMES))

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/scripts/process_dataset.py"
# --stanford_only --force_reprocess
# !python process_dataset.py --force_reprocess
# !python process_dataset.py --only_splits

In [ ]:
!python "/content/drive/MyDrive/Colab Notebooks/Pawnder/scripts/stanford_keypoint_processor.py"

In [ ]:
# Keypoint Updater
import os
import json
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("KeypointUpdater")

def load_keypoint_annotations(keypoint_dir):
    """Load keypoint annotations from files"""
    logger.info(f"Loading keypoint annotations from {keypoint_dir}...")

    if not os.path.exists(keypoint_dir):
        logger.warning(f"Keypoint directory not found: {keypoint_dir}")
        return {}

    # Load all keypoint files
    keypoints = {}
    keypoint_files = glob.glob(os.path.join(keypoint_dir, "*.json"))

    for kp_file in tqdm(keypoint_files, desc="Loading keypoint files"):
        try:
            with open(kp_file, 'r') as f:
                keypoint_data = json.load(f)

            # Extract the image ID from filename
            filename = os.path.basename(kp_file)
            image_id = os.path.splitext(filename)[0]

            # Store keypoints indexed by image ID
            keypoints[image_id] = keypoint_data

        except Exception as e:
            logger.warning(f"Error loading keypoint file {kp_file}: {str(e)}")

    logger.info(f"Loaded keypoints for {len(keypoints)} images")
    return keypoints

def update_annotations_with_keypoints(annotations_path, keypoints, output_path=None):
    """Update annotations CSV with keypoint data"""
    logger.info(f"Updating annotations at {annotations_path} with keypoints...")

    # Load annotations
    try:
        annotations = pd.read_csv(annotations_path)
    except Exception as e:
        logger.error(f"Error loading annotations: {e}")
        return None

    # Add keypoint columns
    annotations['has_keypoints'] = False
    annotations['keypoint_file'] = None

    # Track matches
    match_count = 0

    # Update each row
    for idx, row in tqdm(annotations.iterrows(), total=len(annotations), desc="Matching keypoints"):
        # Get image path
        if 'image_path' not in row:
            continue

        image_path = row['image_path']

        # Try different naming patterns to match with keypoints
        image_id = os.path.splitext(os.path.basename(image_path))[0]

        # Direct match
        if image_id in keypoints:
            annotations.at[idx, 'has_keypoints'] = True
            annotations.at[idx, 'keypoint_file'] = f"{image_id}.json"
            match_count += 1
            continue

        # Try with 'stanford_' prefix removed (if present)
        if image_id.startswith('stanford_'):
            stanford_id = image_id[9:]  # Remove 'stanford_' prefix
            if stanford_id in keypoints:
                annotations.at[idx, 'has_keypoints'] = True
                annotations.at[idx, 'keypoint_file'] = f"{stanford_id}.json"
                match_count += 1
                continue

        # Try with original ID
        if 'original_id' in row and not pd.isna(row['original_id']):
            original_id = os.path.splitext(os.path.basename(row['original_id']))[0]
            if original_id in keypoints:
                annotations.at[idx, 'has_keypoints'] = True
                annotations.at[idx, 'keypoint_file'] = f"{original_id}.json"
                match_count += 1
                continue

    logger.info(f"Matched keypoints for {match_count} out of {len(annotations)} annotations ({match_count/len(annotations)*100:.1f}%)")

    # Save updated annotations if output path provided
    if output_path:
        annotations.to_csv(output_path, index=False)
        logger.info(f"Saved updated annotations to {output_path}")

    return annotations

# Define your directories explicitly
base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
keypoint_dir = f"{base_dir}/Data/annotations/stanford_keypoints"
# Set to the processed directory which contains train, validation, test folders
annotations_dir = f"{base_dir}/Data/processed"

# Verify directories exist
if not os.path.exists(keypoint_dir):
    logger.error(f"Keypoint directory not found: {keypoint_dir}")
else:
    # Load keypoints
    keypoints = load_keypoint_annotations(keypoint_dir)

    if not keypoints:
        logger.error("No keypoints loaded. Exiting.")
    else:
        # Update annotations for each split
        for split in ['train', 'validation', 'test']:
            split_dir = os.path.join(annotations_dir, split)
            if not os.path.exists(split_dir):
                logger.warning(f"Split directory not found: {split_dir}")
                continue

            annotations_path = os.path.join(split_dir, 'annotations.csv')
            if not os.path.exists(annotations_path):
                logger.warning(f"Annotations file not found: {annotations_path}")
                continue

            # Update annotations
            updated = update_annotations_with_keypoints(
                annotations_path,
                keypoints,
                output_path=annotations_path  # Overwrite original
            )

            if updated is not None:
                logger.info(f"Successfully updated {split} annotations")

# Model Architecture

In [ ]:
# Import the fixed model
from dog_emotion_with_behaviors_fixed import DogEmotionWithBehaviors, find_directory

# Try to auto-detect the correct directory
train_dir = find_directory()
if train_dir:
    # Use the parent directory as the processed_dir
    processed_dir = os.path.dirname(train_dir)
    # Create classifier with the found directory
    classifier = DogEmotionWithBehaviors(base_dir=os.path.dirname(processed_dir))
else:
    # Use default directory
    classifier = DogEmotionWithBehaviors()

# Train with fewer epochs for testing
history, model_dir = classifier.train(
    epochs=10,               # Start with fewer epochs to test
    batch_size=32,
    img_size=(224, 224),
    fine_tune=False          # Start without fine-tuning for faster testing
)

# Train Model

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/behavior_matrix_processor.py"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/dog_emotion_demo.py" --train --epochs 50 --batch_size 32

# Debugging

In [ ]:
!python model_execution.py prepare

In [ ]:
!python model_execution.py diagnose

In [ ]:
import os
import sys

def test_model_imports():
    print("Testing model imports...")

    try:
        from dog_emotion_with_behaviors_fixed import DogEmotionWithBehaviors
        classifier = DogEmotionWithBehaviors()
        print("✓ Successfully imported model")
        return True
    except Exception as e:
        print(f"✗ Import error: {str(e)}")
        return False

def test_data_loading():
    print("\nTesting data loading...")

    try:
        from dog_emotion_with_behaviors_fixed import DogEmotionWithBehaviors
        classifier = DogEmotionWithBehaviors()
        train_annotations = classifier.load_annotations('train')

        if train_annotations:
            print(f"✓ Successfully loaded {len(train_annotations)} train annotations")
            return True
        else:
            print("✗ Failed to load train annotations")
            return False
    except Exception as e:
        print(f"✗ Error loading data: {str(e)}")
        return False

# Run tests
import_ok = test_model_imports()
data_ok = import_ok and test_data_loading()

print("\nTest Results:")
print(f"  Model imports: {'✓ Success' if import_ok else '✗ Failed'}")
print(f"  Data loading:  {'✓ Success' if data_ok else '✗ Failed'}")

In [ ]:
# Import the model
from dog_emotion_with_behaviors_fixed import DogEmotionWithBehaviors

# Create classifier
classifier = DogEmotionWithBehaviors()

# Train the model (you can use fewer epochs for testing)
history, model_dir = classifier.train(
    epochs=10,               # Start with fewer epochs for testing
    batch_size=32,
    img_size=(224, 224),
    fine_tune=False          # Disable fine-tuning for faster testing
)

# Post Model Trained

In [ ]:
# Analyzing Images
# Load a pre-trained model
classifier = DogEmotionWithBehaviors()
classifier.load_model("path/to/model.h5")

# Analyze an image
result = classifier.predict_image("path/to/dog_image.jpg")

# Display results
print(f"Predicted emotion: {result['emotion']} (Score: {result['emotion_score']:.2f})")
print(f"Confidence: {result['confidence']:.2f}")

# Visualize the prediction
classifier.visualize_prediction("path/to/dog_image.jpg", result)

# Training A New Model

In [ ]:
# Create the classifier
classifier = DogEmotionWithBehaviors()

# Train the model
history, model_dir = classifier.train(
    epochs=50,
    batch_size=32,
    fine_tune=True
)

print(f"Model saved to {model_dir}")